# M16 · Arch 4C loss sweep (pick which loss to KEEP)

Frozen architecture = **Arch 4C** (full LN + prototype + private `(m_g, ρ_g)`).
Only the **extra loss term** changes. Compare against L0 = M15c baseline.

| id | name | extra term |
|---|---|---|
| `L0` | baseline | none (M15c loss) |
| `PSC` | Private Score Coupling | `(m_g − α ρ_g − γ)²` |
| `CGSS` | Cross-Gen Score Separation | score-bank push across gens (batch-safe) |
| `ARSD` | Asymmetric Real–Fake Drift | shared `m_real` + private fake margin |
| `BGTC` | Batch–Gen Temp Calibration | private-τ sigmoid calibration |

**KEEP a variant** iff NT ≥ 81.21 and DFD ≥ 95.5, then rank by NT then DFD vs L0.
Set `LOSS_IDS` to run a subset. Results under `runs/m16/<loss_id>/`.


## 1. Setup
### 1.1 Settings


In [1]:
# 1.1 Settings
import json, os
from pathlib import Path

DATA_ROOT = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection")
FACES_ROOT = DATA_ROOT / "faces"
INDEX_CSV = DATA_ROOT / "index" / "ffpp_c23.csv"
FACES_DFD = DATA_ROOT / "faces_dfd"
DFD_INDEX = DATA_ROOT / "index" / "dfd_c23.csv"
FACES_UADFV = DATA_ROOT / "faces_uadfv"
UADFV_INDEX = DATA_ROOT / "index" / "uadfv.csv"
RUNS_ROOT = DATA_ROOT / "runs"
M16_DIR = RUNS_ROOT / "m16"
M15C_DIR = RUNS_ROOT / "m15c"
M4_DIR = RUNS_ROOT / "m4"

ALL_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
TRAIN_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
SHORT = {"original": "real", "Deepfakes": "DF", "Face2Face": "F2F", "FaceSwap": "FS",
         "NeuralTextures": "NT", "FaceShifter": "FSh"}
G = len(TRAIN_GENERATORS)

ARCH, PRETRAINED = "ViT-L-14", "openai"
L2_NORM = True
W_UNIFORM, W_ALIGN = 0.5, 0.1
AUGMENT = True
BATCH_SIZE = 128
EPOCHS = 10
CYCLE_EPOCHS = 10
WARMUP_EPOCHS = 1
LR_MAX, LR_MIN = 3e-4, 1e-5
SEED = 0
LAMBDA = 0.01
BETA_MARGIN = 1.0
MARGIN0 = 0.1
EMA = 0.99
TAU_INIT = 0.1
PROTO_WARM = 512
RUN_DFD, RUN_UADFV = True, True

# --- loss-sweep knobs (architecture frozen) ---
LOSS_IDS = ["L0", "PSC", "CGSS", "ARSD", "BGTC"]  # edit to run a subset
KAPPA = {
    "L0": 0.0,
    "PSC": 0.1,
    "CGSS": 0.5,
    "ARSD": 1.0,   # replaces baseline margin when ARSD is on
    "BGTC": 0.5,
}
PSC_ALPHA, PSC_GAMMA = 0.5, 0.0
CGSS_ETA = 0.15          # min |score_mean_g - score_mean_g'|
CGSS_EMA = 0.95
ARSD_M_REAL0 = 0.3
BGTC_EPS = 0.1           # push real→eps, fake→1-eps under private τ

A0 = {"seen": 94.54, "DF": 99.49, "F2F": 97.86, "FS": 99.60, "NT": 81.21, "FSh": 92.25, "DFD": 95.15}
REFS = {
    "full LayerNorms (M4 λ=∞)": (M4_DIR / f"gbla_laminf_{ARCH}_seed{SEED}", 96.05, 99.38),
    "GBLA λ=0.01 (M4) / Arch1": (M4_DIR / f"gbla_lam0.01_{ARCH}_seed{SEED}", 96.71, 99.63),
    "Arch4C L0 (M15c)": (M15C_DIR / f"m15c_arch4c_{ARCH}_seed{SEED}", None, None),
}
VAL_FRAMES_PER_VIDEO = 8
CACHE_IN_RAM = True
EVAL_BATCH = 256
GRAD_CHECKPOINTING = False

COMMON_CONFIG = {
    "architecture": "Arch4C", "notebook": "M16",
    "private": "score-space (m_g, rho_g) train only",
    "docs": "docs4/arch4c_score_space.tex",
    "loss_sweep": LOSS_IDS,
    "kappa": KAPPA,
    "lambda": LAMBDA, "beta_margin": BETA_MARGIN, "margin0": MARGIN0,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "seed": SEED,
    "lr_max": LR_MAX, "lr_min": LR_MIN, "augment": AUGMENT, "arch": ARCH,
}
assert INDEX_CSV.exists()
M16_DIR.mkdir(parents=True, exist_ok=True)
print("M16 loss sweep ->", M16_DIR)
print("LOSS_IDS:", LOSS_IDS)


M16 loss sweep -> /nlsasfs/home/isea/isea64/GBLA-Deepfake Detection/runs/m16
LOSS_IDS: ['L0', 'PSC', 'CGSS', 'ARSD', 'BGTC']


### 1.2 Imports


In [2]:
# 1.2 Imports and environment
import json, math, random, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, read_image
from torchvision.transforms import v2 as T
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
assert torch.cuda.is_available(), "M16 needs the GPU"
DEVICE = "cuda"


def usable_cpus():
    """CPUs this job may use: the affinity mask and the cgroup CPU quota."""
    n = len(os.sched_getaffinity(0))
    for quota_file, period_file in [("/sys/fs/cgroup/cpu.max", None),
                                    ("/sys/fs/cgroup/cpu/cpu.cfs_quota_us", "/sys/fs/cgroup/cpu/cpu.cfs_period_us")]:
        try:
            if period_file is None:
                quota, period = Path(quota_file).read_text().split()
            else:
                quota, period = Path(quota_file).read_text().strip(), Path(period_file).read_text().strip()
            if quota not in ("max", "-1"):
                n = min(n, max(1, int(quota) // int(period)))
        except (OSError, ValueError):
            pass
    return n


def available_memory_gb():
    """Free RAM for this job: the smaller of MemAvailable and the cgroup limit minus non-cache usage."""
    candidates = []
    try:
        meminfo = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
        candidates.append(int(meminfo["MemAvailable"].split()[0]) * 1024)
    except (OSError, KeyError, ValueError):
        pass
    for limit_file, usage_file, stat_file, cache_key in [
            ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.stat", "file"),
            ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes",
             "/sys/fs/cgroup/memory/memory.stat", "cache")]:
        try:
            limit = Path(limit_file).read_text().strip()
            if not limit.isdigit() or int(limit) >= 1 << 60:
                continue
            usage = int(Path(usage_file).read_text().strip())
            stat = dict(line.split() for line in Path(stat_file).read_text().splitlines())
            candidates.append(int(limit) - usage + int(stat.get(cache_key, 0)))
        except (OSError, ValueError):
            pass
    return min(candidates) / 1e9 if candidates else 0.0


CPUS = usable_cpus()
NUM_WORKERS = max(1, min(16, CPUS - 1))     # one CPU stays with the training process
torch.set_num_threads(2)

pd.Series({
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "open_clip": open_clip.__version__,
    "GPU": torch.cuda.get_device_name(0),
    "CPU cores on the node": os.cpu_count(),
    "CPUs usable by this job": CPUS,
    "data loader workers": NUM_WORKERS,
    "free RAM for this job (GB)": round(available_memory_gb(), 1),
}, name="value").to_frame()


/nlsasfs/home/isea/isea64/venv_jupyter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
torch,2.6.0+cu124
torchvision,0.21.0+cu124
open_clip,3.3.0
GPU,NVIDIA A100-SXM4-40GB
CPU cores on the node,256
CPUs usable by this job,8
data loader workers,7
free RAM for this job (GB),953.1


## 2. Data
### 2.1


In [3]:
# 2.1 Index and subsets
index = pd.read_csv(INDEX_CSV, dtype={"video_id": str, "target_id": str, "source_id": str},
                    keep_default_na=False)
CATEGORIES = ["original"] + ALL_GENERATORS
GEN_CODE = {g: k for k, g in enumerate(CATEGORIES)}


def subset(split, generators):
    keep = (index.split == split) & index.generator.isin(["original", *generators])
    return index[keep].reset_index(drop=True)


train_df = subset("train", TRAIN_GENERATORS)
val_all = subset("val", TRAIN_GENERATORS)
frames_per_video = int(val_all.groupby(["generator", "video_id"]).size().max())
keep_every = max(1, frames_per_video // VAL_FRAMES_PER_VIDEO)
frame_rank = val_all.groupby(["generator", "video_id"]).frame_idx.rank(method="first").astype(int) - 1
val_df = val_all[frame_rank % keep_every == 0].reset_index(drop=True)
test_df = subset("test", ALL_GENERATORS)

display(pd.DataFrame({name: df.groupby("generator").size().reindex(CATEGORIES)
                      for name, df in [("train", train_df), ("val (subsampled)", val_df), ("test", test_df)]}
                     ).fillna(0).astype(int))


,train,val (subsampled),test
generator,,,
original,23040,1120,4480
Deepfakes,23040,1120,4480
Face2Face,23040,1120,4480
FaceSwap,23040,1120,4480
NeuralTextures,23040,1120,4480
FaceShifter,0,0,4480


### 2.2


In [4]:
# 2.2 Decoded crops in RAM
def load_images(df, desc):
    """Decodes every crop of an index slice into one uint8 array (N, H, W, 3), using threads."""
    paths = [FACES_ROOT / p for p in df.path]
    first = np.asarray(Image.open(paths[0]).convert("RGB"))
    out = np.empty((len(paths), *first.shape), dtype=np.uint8)

    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))

    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out


images = {}
budget = 0.6 * available_memory_gb()
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    need = len(df) * 224 * 224 * 3 / 1e9
    if CACHE_IN_RAM and need <= budget:
        t = time.time()
        images[name] = load_images(df, f"{name} crops -> RAM")
        budget -= need
        sec = time.time() - t
        print(f"{name}: {len(df):,} crops, {need:.1f} GB in RAM, {sec / 60:.1f} min ({len(df) / sec:.0f} crops/s from disk)")
    else:
        images[name] = None
        print(f"{name}: read from disk during training (needs {need:.1f} GB, RAM budget left {budget:.1f} GB)")


train crops -> RAM: 100%|██████████| 115200/115200 [01:09<00:00, 1668.53it/s]


train: 115,200 crops, 17.3 GB in RAM, 1.2 min (1596 crops/s from disk)


val crops -> RAM: 100%|██████████| 5600/5600 [00:03<00:00, 1689.02it/s]


val: 5,600 crops, 0.8 GB in RAM, 0.1 min (1639 crops/s from disk)


test crops -> RAM: 100%|██████████| 26880/26880 [00:15<00:00, 1727.82it/s]

test: 26,880 crops, 4.0 GB in RAM, 0.3 min (1688 crops/s from disk)


### 2.3


In [5]:
# 2.3 Round-robin loader
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def make_transform(train):
    steps = []
    if train and AUGMENT:
        steps += [
            T.RandomHorizontalFlip(0.5),
            T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))], p=0.1),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.JPEG(quality=(40, 100)),
        ]
    steps += [T.ToDtype(torch.float32, scale=True), T.Normalize(CLIP_MEAN, CLIP_STD)]
    return T.Compose(steps)


class FaceDataset(Dataset):
    def __init__(self, df, train, images=None):
        self.paths = [str(FACES_ROOT / p) for p in df.path]
        self.images = images
        self.labels = torch.tensor(df.label.values, dtype=torch.long)
        self.gens = torch.tensor(df.generator.map(GEN_CODE).values, dtype=torch.long)
        self.transform = make_transform(train)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        if self.images is not None:
            image = torch.from_numpy(np.array(self.images[i])).permute(2, 0, 1)
        else:
            image = read_image(self.paths[i], mode=ImageReadMode.RGB)
        return self.transform(tv_tensors.Image(image)).as_subclass(torch.Tensor), self.labels[i], self.gens[i]


class RoundRobinSampler(Sampler):
    """One epoch = one block per training generator, in an order fixed by (seed, epoch)."""

    def __init__(self, df, seed):
        self.seed = seed
        self.epoch = 0
        gen = df.generator.values
        real_idx = np.flatnonzero(gen == "original")
        # fixed split: real row r goes to generator r % G
        self.blocks = {}
        for k, g in enumerate(TRAIN_GENERATORS):
            fakes = np.flatnonzero(gen == g)
            reals = real_idx[k::G][: len(fakes) // G]
            self.blocks[g] = (fakes, reals)
        self.n_per = len(self.blocks[TRAIN_GENERATORS[0]][0]) + len(self.blocks[TRAIN_GENERATORS[0]][1])

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        order_rng = np.random.default_rng([self.seed, self.epoch, 1])
        gens = list(TRAIN_GENERATORS)
        order_rng.shuffle(gens)
        out = []
        for g in gens:
            fakes, reals = self.blocks[g]
            rng = np.random.default_rng([self.seed, ALL_GENERATORS.index(g), self.epoch])
            out.extend(rng.permutation(np.concatenate([fakes, reals])).tolist())
        return iter(out)

    def __len__(self):
        return self.n_per * G


train_ds = FaceDataset(train_df, train=True, images=images["train"])
train_sampler = RoundRobinSampler(train_df, SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
owned = {SHORT[g]: (len(f), len(r)) for g, (f, r) in train_sampler.blocks.items()}
print("fakes / reals owned by each generator:", owned)
print(f"steps per epoch: {len(train_loader)}  (M1 was 900);  workers: {NUM_WORKERS}")
assert len(train_loader) == 900, len(train_loader)


fakes / reals owned by each generator: {'DF': (23040, 5760), 'F2F': (23040, 5760), 'FS': (23040, 5760), 'NT': (23040, 5760)}
steps per epoch: 900  (M1 was 900);  workers: 7


### 2.4


In [6]:
# 2.4 Data-loading speed
def loader_rate(loader, n_batches=30):
    """Steady-state images/s after the pre-fetched batches are used up."""
    it = iter(loader)
    for _ in range(NUM_WORKERS * 4 + 2):
        next(it)
    t = time.time()
    for _ in range(n_batches):
        next(it)
    rate = n_batches * loader.batch_size / (time.time() - t)
    del it
    return rate


t = time.time()
for i in range(200):
    train_ds[i]
ms_per_crop = (time.time() - t) / 200 * 1000
load_speed = loader_rate(train_loader)
GPU_SPEED_M1 = 253
speed = min(load_speed, GPU_SPEED_M1)
pd.Series({
    "CPU time per augmented crop, one process (ms)": round(ms_per_crop, 1),
    "data loading speed (img/s)": round(load_speed),
    "GPU training speed in M1 (img/s)": GPU_SPEED_M1,
    "bottleneck": "GPU" if load_speed >= GPU_SPEED_M1 else "data loading",
    "estimated minutes per epoch (training part)": round(len(train_loader) * BATCH_SIZE / speed / 60, 1),
    f"estimated hours per ablation ({EPOCHS} epochs)": round(EPOCHS * len(train_loader) * BATCH_SIZE / speed / 3600, 1),
}, name="value").to_frame()


,value
"CPU time per augmented crop, one process (ms)",20.6
data loading speed (img/s),1047
GPU training speed in M1 (img/s),253
bottleneck,GPU
estimated minutes per epoch (training part),7.6
estimated hours per ablation (10 epochs),1.3


## 3. Model
### 3.1 Detector (LN only)


In [7]:
# 3.1 Detector — full LN tune, linear head unused
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        enc = open_clip.create_model(ARCH, pretrained=PRETRAINED, force_quick_gelu=True).visual
        enc.proj = None
        self.encoder = enc
        self.head = nn.Linear(enc.transformer.width, 2)

    def forward(self, x):
        feats = self.encoder(x).float()
        f = F.normalize(feats, dim=-1)
        return self.head(f if L2_NORM else feats), f

torch.manual_seed(SEED)
model = Detector().to(DEVICE)
model.requires_grad_(False)
for m in model.encoder.modules():
    if isinstance(m, nn.LayerNorm):
        m.requires_grad_(True)
model.head.requires_grad_(False)  # prototype head instead
model.encoder.set_grad_checkpointing(GRAD_CHECKPOINTING)
PARAMS = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
PARAM_NAMES = [n for n, _ in PARAMS]
P = sum(p.numel() for _, p in PARAMS)

def get_flat():
    return torch.cat([p.detach().reshape(-1) for _, p in PARAMS]).float()

@torch.no_grad()
def set_flat(theta):
    offset = 0
    for _, p in PARAMS:
        n = p.numel(); p.copy_(theta[offset:offset + n].view_as(p)); offset += n

THETA0 = get_flat().clone()
print(f"LN-only trainable P = {P:,} (no linear head)")
assert P == 102_400



LN-only trainable P = 102,400 (no linear head)


### 3.2 Arch4C state (+ score bank / m_real)


In [8]:
# 3.2 Arch4C state: full LN + score-space private (+ optional m_real, score bank)
WIDTH = model.encoder.transformer.width

class Arch4CState:
    def __init__(self, seed=SEED):
        self.b = torch.zeros((), device=DEVICE)
        self.rho = torch.full((), math.log(TAU_INIT), device=DEVICE)  # shared deploy temp
        self.m = {g: torch.tensor(MARGIN0, device=DEVICE) for g in TRAIN_GENERATORS}
        self.rho_g = {g: torch.full((), math.log(TAU_INIT), device=DEVICE) for g in TRAIN_GENERATORS}
        self.m_real = torch.tensor(ARSD_M_REAL0, device=DEVICE)  # shared; used by ARSD only
        self.proto = {"real": torch.zeros(WIDTH, device=DEVICE), "n_real": 0,
                      "fake": torch.zeros(WIDTH, device=DEVICE), "n_fake": 0}
        self.theta = THETA0.clone()
        # EMA of mean raw scores s=(z-mbar)·w per generator (for CGSS; train-only)
        self.score_ema = {g: torch.zeros((), device=DEVICE) for g in TRAIN_GENERATORS}
        self.score_ema_n = {g: 0 for g in TRAIN_GENERATORS}

    def head_parts(self):
        assert self.proto["n_real"] > 0 and self.proto["n_fake"] > 0
        cr = F.normalize(self.proto["real"], dim=0)
        cf = F.normalize(self.proto["fake"], dim=0)
        return {"w": F.normalize(cf - cr, dim=0), "mbar": (cr + cf) / 2, "c_real": cr, "c_fake": cf}

    @torch.no_grad()
    def update_prototypes(self, z, y):
        def ema(old, new, n):
            return new.clone() if n == 0 else EMA * old + (1 - EMA) * new
        real = y == 0
        if real.any():
            self.proto["real"] = ema(self.proto["real"], z[real].mean(0), self.proto["n_real"]); self.proto["n_real"] += 1
        if (~real).any():
            self.proto["fake"] = ema(self.proto["fake"], z[~real].mean(0), self.proto["n_fake"]); self.proto["n_fake"] += 1

    @torch.no_grad()
    def update_score_ema(self, g, s_mean):
        n = self.score_ema_n[g]
        if n == 0:
            self.score_ema[g].copy_(s_mean)
        else:
            self.score_ema[g].mul_(CGSS_EMA).add_(s_mean, alpha=1 - CGSS_EMA)
        self.score_ema_n[g] = n + 1

    def save(self):
        return {"architecture": "Arch4C",
                "theta": self.theta.detach().cpu(),
                "b": self.b.detach().cpu(), "rho": self.rho.detach().cpu(),
                "m": {g: v.detach().cpu() for g, v in self.m.items()},
                "rho_g": {g: v.detach().cpu() for g, v in self.rho_g.items()},
                "m_real": self.m_real.detach().cpu(),
                "score_ema": {g: v.detach().cpu() for g, v in self.score_ema.items()},
                "score_ema_n": dict(self.score_ema_n),
                "proto": {"real": self.proto["real"].cpu(), "n_real": self.proto["n_real"],
                          "fake": self.proto["fake"].cpu(), "n_fake": self.proto["n_fake"]}}

    def load(self, obj):
        self.theta.copy_(obj["theta"].to(DEVICE))
        self.b.copy_(obj["b"].to(DEVICE)); self.rho.copy_(obj["rho"].to(DEVICE))
        for g in TRAIN_GENERATORS:
            self.m[g].copy_(obj["m"][g].to(DEVICE)); self.rho_g[g].copy_(obj["rho_g"][g].to(DEVICE))
        if "m_real" in obj:
            self.m_real.copy_(obj["m_real"].to(DEVICE))
        if "score_ema" in obj:
            for g in TRAIN_GENERATORS:
                self.score_ema[g].copy_(obj["score_ema"][g].to(DEVICE))
                self.score_ema_n[g] = int(obj["score_ema_n"][g])
        self.proto["real"].copy_(obj["proto"]["real"].to(DEVICE))
        self.proto["fake"].copy_(obj["proto"]["fake"].to(DEVICE))
        self.proto["n_real"] = int(obj["proto"]["n_real"]); self.proto["n_fake"] = int(obj["proto"]["n_fake"])

deployed = P + 2
train_n = deployed + G * 2 + 1  # +1 m_real (train-only; unused at deploy)
display(pd.DataFrame([
    {"piece": "full LNs", "n": P, "dep": True},
    {"piece": "prototype b, rho", "n": 2, "dep": True},
    {"piece": "private (m_g, rho_g)", "n": G * 2, "dep": False},
    {"piece": "m_real (ARSD only)", "n": 1, "dep": False},
    {"piece": "TOTAL deployed", "n": deployed, "dep": True},
]))
assert deployed < 104_450
print("deployed", deployed, "train (max)", train_n)


,piece,n,dep
0,full LNs,102400,True
1,"prototype b, rho",2,True
2,"private (m_g, rho_g)",8,False
3,m_real (ARSD only),1,False
4,TOTAL deployed,102402,True


deployed 102402 train (max) 102411


## 4. Helpers (parameterized loss)


In [9]:
# 4.1 Helpers — Arch4C encode/eval + parameterized loss
MEAN_GPU = torch.tensor(CLIP_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_GPU = torch.tensor(CLIP_STD, device=DEVICE).view(1, 3, 1, 1)

def encode(x):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        feats = model.encoder(x)
    return F.normalize(feats.float(), dim=-1)

def head_logit(z, b, rho, hp):
    return ((z - hp["mbar"]) @ hp["w"] - b) / rho.exp()

def raw_score(z, hp):
    return (z - hp["mbar"]) @ hp["w"]

def arch4c_loss(z, y, g, state, b, rho_g, m_g, m_real, loss_id):
    """L0 baseline + optional extra term selected by loss_id."""
    hp = state.head_parts()
    logits = torch.stack([torch.zeros_like(z[:, 0]), head_logit(z, b, rho_g, hp)], dim=1)
    ce_each = F.cross_entropy(logits, y, reduction="none")
    ce = ce_each.mean()
    n = len(y)
    i, j = torch.triu_indices(n, n, offset=1, device=z.device)
    d2 = (2 - 2 * (z @ z.T)[i, j]).clamp(min=0)
    same = y[i] == y[j]
    align = d2[same].mean() if same.any() else d2.new_zeros(())
    uniform = torch.exp(-2 * d2).mean().clamp(min=1e-6).log()
    cr = hp["c_real"]
    real = y == 0
    fake = ~real

    # baseline margin (used by L0 / PSC / CGSS / BGTC; ARSD replaces it)
    if loss_id == "ARSD":
        l_real = F.relu(m_real - (z[real] @ cr)).mean() if real.any() else z.new_zeros(())
        l_fake = F.relu((z[fake] @ cr) - m_g).mean() if fake.any() else z.new_zeros(())
        l_margin = l_real + l_fake
        beta = KAPPA["ARSD"]
    else:
        l_margin = F.relu((z[fake] @ cr) - m_g).mean() if fake.any() else z.new_zeros(())
        beta = BETA_MARGIN

    penalty = LAMBDA * (m_g.pow(2) + rho_g.pow(2))
    extra = z.new_zeros(())

    if loss_id == "PSC":
        # couple private margin and log-temp of the active generator
        extra = (m_g - PSC_ALPHA * rho_g - PSC_GAMMA).pow(2)
    elif loss_id == "CGSS":
        # push active gen mean raw score away from other gens' EMA scores
        s = raw_score(z, hp)
        s_mean = s[fake].mean() if fake.any() else s.mean()
        gaps = []
        for gp in TRAIN_GENERATORS:
            if gp == g or state.score_ema_n[gp] == 0:
                continue
            gaps.append(F.relu(CGSS_ETA - (s_mean - state.score_ema[gp].detach()).abs()))
        if gaps:
            extra = torch.stack(gaps).mean()
        # score bank update happens in train_step (no grad)
    elif loss_id == "BGTC":
        # calibrate private τ so batch probs sit near (eps, 1-eps)
        p = torch.sigmoid(head_logit(z, b, rho_g, hp))
        if real.any():
            extra = extra + (p[real].mean() - BGTC_EPS).pow(2)
        if fake.any():
            extra = extra + (p[fake].mean() - (1.0 - BGTC_EPS)).pow(2)
    # L0 / ARSD: extra stays 0 (ARSD already in l_margin)

    kappa = float(KAPPA.get(loss_id, 0.0)) if loss_id not in ("L0", "ARSD") else 0.0
    if loss_id == "ARSD":
        kappa = 0.0  # weight already via beta=KAPPA["ARSD"]

    loss = ce + W_ALIGN * align + W_UNIFORM * uniform + beta * l_margin + penalty + kappa * extra
    parts = {
        "loss": float(loss.detach()), "ce": float(ce.detach()),
        "margin": float(l_margin.detach()), "penalty": float(penalty.detach()),
        "extra": float(extra.detach()),
    }
    return loss, parts, ce_each.detach(), (raw_score(z, hp).detach() if loss_id == "CGSS" else None)

def lr_at(step, steps_per_epoch):
    cycle = CYCLE_EPOCHS * steps_per_epoch
    warm = WARMUP_EPOCHS * steps_per_epoch
    pos = step % cycle
    if pos < warm:
        return LR_MIN + (LR_MAX - LR_MIN) * pos / warm
    frac = (pos - warm) / max(cycle - warm, 1)
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * frac))

def adam_step(param, grad, m, v, t, lr):
    b1, b2, eps = 0.9, 0.999, 1e-8
    t += 1
    m.mul_(b1).add_(grad, alpha=1 - b1)
    v.mul_(b2).addcmul_(grad, grad, value=1 - b2)
    param.addcdiv_(m / (1 - b1 ** t), (v / (1 - b2 ** t)).sqrt().add_(eps), value=-lr)

class AdamSlots:
    @staticmethod
    def tensors(state, loss_id="L0"):
        out = {"theta": state.theta, "b": state.b, "rho": state.rho}
        out.update({f"m:{g}": state.m[g] for g in TRAIN_GENERATORS})
        out.update({f"rho_g:{g}": state.rho_g[g] for g in TRAIN_GENERATORS})
        if loss_id == "ARSD":
            out["m_real"] = state.m_real
        return out
    def __init__(self, state, loss_id="L0"):
        self.loss_id = loss_id
        ts = self.tensors(state, loss_id)
        self.m = {k: torch.zeros_like(v) for k, v in ts.items()}
        self.v = {k: torch.zeros_like(v) for k, v in ts.items()}
        self.t = {k: torch.zeros((), dtype=torch.long, device=DEVICE) for k in ts}
    def state_dict(self):
        return {"m": {k: v.cpu() for k, v in self.m.items()}, "v": {k: v.cpu() for k, v in self.v.items()},
                "t": {k: int(v) for k, v in self.t.items()}, "loss_id": self.loss_id}
    def load_state_dict(self, s):
        for k in self.m:
            self.m[k].copy_(s["m"][k].to(DEVICE)); self.v[k].copy_(s["v"][k].to(DEVICE)); self.t[k].fill_(s["t"][k])

def flat_grad_safe():
    return torch.cat([(p.grad if p.grad is not None else torch.zeros_like(p)).reshape(-1) for _, p in PARAMS]).float()

def train_step(state, opt, x, y, g, lr, loss_id):
    set_flat(state.theta)
    model.zero_grad(set_to_none=True)
    b = state.b.detach().clone().requires_grad_(True)
    rho_g = state.rho_g[g].detach().clone().requires_grad_(True)
    m_g = state.m[g].detach().clone().requires_grad_(True)
    m_real = state.m_real.detach().clone().requires_grad_(loss_id == "ARSD")
    z = encode(x)
    loss, parts, ce_each, s_det = arch4c_loss(z, y, g, state, b, rho_g, m_g, m_real, loss_id)
    loss.backward()
    g_theta = flat_grad_safe().detach()
    grads = {"theta": g_theta, "b": b.grad.detach(), f"rho_g:{g}": rho_g.grad.detach(), f"m:{g}": m_g.grad.detach()}
    if loss_id == "ARSD" and m_real.grad is not None:
        grads["m_real"] = m_real.grad.detach()
    state.update_prototypes(z.detach(), y)
    if loss_id == "CGSS" and s_det is not None:
        fake = y == 1
        if fake.any():
            state.update_score_ema(g, s_det[fake].mean())
    tensors = AdamSlots.tensors(state, loss_id)
    for k, gk in grads.items():
        adam_step(tensors[k], gk, opt.m[k], opt.v[k], opt.t[k], lr)
    with torch.no_grad():
        state.rho.copy_(0.99 * state.rho + 0.01 * state.rho_g[g])
    return parts, ce_each

def to_model_input(uint8_batch):
    x = torch.as_tensor(uint8_batch).to(DEVICE, non_blocking=True)
    return (x.permute(0, 3, 1, 2).float() / 255 - MEAN_GPU) / STD_GPU

def load_paths(paths, desc):
    out = np.empty((len(paths), 224, 224, 3), dtype=np.uint8)
    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))
    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out

def ram_images(name, df):
    if images.get(name) is None:
        images[name] = load_images(df, f"{name} crops -> RAM")
    return images[name]

@torch.no_grad()
def predict_probs(state, arr, desc="predict"):
    set_flat(state.theta); model.eval()
    hp = state.head_parts()
    b, rho = state.b, state.rho
    out = []
    for i in tqdm(range(0, len(arr), EVAL_BATCH), desc=desc, leave=False):
        z = encode(to_model_input(arr[i:i + EVAL_BATCH]))
        out.append(torch.sigmoid(head_logit(z, b, rho, hp)).float().cpu())
    model.train(); return torch.cat(out).numpy()

@torch.no_grad()
def exact_prototypes(state, n_per=None, seed=SEED):
    arr = ram_images("train", train_df)
    set_flat(state.theta); model.eval()
    rng = np.random.default_rng([seed, 31])
    real_rows = np.concatenate([train_sampler.blocks[g][1] for g in TRAIN_GENERATORS])
    fake_rows = np.concatenate([train_sampler.blocks[g][0] for g in TRAIN_GENERATORS])
    proto = {"real": None, "n_real": 1, "fake": None, "n_fake": 1}
    for name, rows in [("real", real_rows), ("fake", fake_rows)]:
        if n_per is not None and len(rows) > n_per:
            rows = np.sort(rng.choice(rows, n_per, replace=False))
        else:
            rows = np.sort(rows)
        total = torch.zeros(WIDTH, device=DEVICE)
        for i in range(0, len(rows), EVAL_BATCH):
            total += encode(to_model_input(arr[rows[i:i + EVAL_BATCH]])).sum(0)
        proto[name] = total / len(rows)
    model.train(); return proto

def auc(labels, scores):
    labels = np.asarray(labels)
    ranks = pd.Series(np.asarray(scores)).rank().values
    n_pos, n_neg = labels.sum(), len(labels) - labels.sum()
    if n_pos == 0 or n_neg == 0: return float("nan")
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def metrics_table(pred):
    groups = {"seen combined (" + "+".join(SHORT[g] for g in TRAIN_GENERATORS) + ")": TRAIN_GENERATORS}
    groups.update({g: [g] for g in ALL_GENERATORS})
    rows = []
    for name, gens in groups.items():
        sub = pred[pred.generator.isin(["original", *gens])]
        if (sub.label == 1).sum() == 0: continue
        vid = sub.groupby(["generator", "video_id"], as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
        rows.append({"test set": name, "status": "seen" if set(gens) <= set(TRAIN_GENERATORS) else "UNSEEN",
                     "frame AUC": 100 * auc(sub.label, sub.prob), "video AUC": 100 * auc(vid.label, vid.prob),
                     "video acc @0.5": 100 * float(((vid.prob > 0.5) == vid.label).mean())})
    return pd.DataFrame(rows).round(2)

def video_auc(index_df, probs):
    vid = index_df.assign(prob=probs).groupby("video_id", as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
    return 100 * auc(vid.label, vid.prob)

def frame_pred(df, probs):
    out = df[["split", "label", "generator", "video_id", "frame_idx", "path"]].copy(); out["prob"] = probs; return out

def save_atomic(obj, path):
    tmp = path.with_name(path.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, path)

def load(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def run_dir(loss_id):
    return M16_DIR / loss_id / f"m16_{loss_id}_{ARCH}_seed{SEED}"

def check_config(rd, config):
    path = rd / "config.json"
    dumped = json.loads(json.dumps(config))
    if path.exists():
        assert json.loads(path.read_text()) == dumped, f"{rd} differs; delete to restart"
    else:
        path.write_text(json.dumps(dumped, indent=1))


### 4.2 Smoke all loss_ids


In [10]:
# 4.2 Smoke — one forward/backward per loss_id
state0 = Arch4CState()
state0.proto = exact_prototypes(state0, n_per=64)
x, y, gen = next(iter(train_loader))
x, y, gen = x.to(DEVICE), y.to(DEVICE), gen.to(DEVICE)
fake_codes = gen[gen != GEN_CODE["original"]]
g = CATEGORIES[int(fake_codes[0])]
rows = []
for lid in LOSS_IDS:
    opt = AdamSlots(state0, lid)
    parts, _ = train_step(state0, opt, x, y, g, LR_MIN, lid)
    rows.append({"loss_id": lid, **parts})
    print(lid, {k: round(v, 4) for k, v in parts.items()})
display(pd.DataFrame(rows))
del state0, opt
torch.cuda.empty_cache()
print("smoke OK")


L0 {'loss': 0.2785, 'ce': 0.6722, 'margin': 0.5016, 'penalty': 0.0531, 'extra': 0.0}
PSC {'loss': 0.432, 'ce': 0.6691, 'margin': 0.5025, 'penalty': 0.0531, 'extra': 1.5657}
CGSS {'loss': 0.2738, 'ce': 0.6659, 'margin': 0.5039, 'penalty': 0.0531, 'extra': 0.0}
ARSD {'loss': 0.2721, 'ce': 0.6633, 'margin': 0.505, 'penalty': 0.0531, 'extra': 0.0}
BGTC {'loss': 0.4137, 'ce': 0.6601, 'margin': 0.5061, 'penalty': 0.0531, 'extra': 0.2876}


,loss_id,loss,ce,margin,penalty,extra
0,L0,0.278465,0.672191,0.501590,0.053119,0.000000
1,PSC,0.432014,0.669090,0.502525,0.053119,1.565746
2,CGSS,0.273836,0.665934,0.503899,0.053118,0.000000
3,ARSD,0.272113,0.663262,0.504961,0.053118,0.000000
4,BGTC,0.413685,0.660053,0.506067,0.053117,0.287635


smoke OK


## 5. Train loss sweep


In [11]:
# 5.1 Train each loss_id (Arch4C frozen; resume-safe per run dir)
def reset_model_to_theta0():
    set_flat(THETA0.clone())
    model.zero_grad(set_to_none=True)

def train_one(loss_id):
    rd = run_dir(loss_id); rd.mkdir(parents=True, exist_ok=True)
    cfg = {**COMMON_CONFIG, "loss_id": loss_id, "kappa_active": KAPPA[loss_id],
           "deployed_trainable": P + 2, "train_trainable": P + 2 + G * 2 + (1 if loss_id == "ARSD" else 0)}
    check_config(rd, cfg)
    reset_model_to_theta0()
    state = Arch4CState(); opt = AdamSlots(state, loss_id); history, start_epoch = [], 0
    if (rd / "last.pt").exists():
        ckpt = load(rd / "last.pt"); state.load(ckpt["state"])
        opt = AdamSlots(state, loss_id); opt.load_state_dict(ckpt["adam"])
        history, start_epoch = ckpt["history"], ckpt["epoch"]; print(loss_id, "resumed", start_epoch)
    else:
        state.proto = exact_prototypes(state, n_per=PROTO_WARM)
    print(f"===== M16 {loss_id} dep={P+2:,} {start_epoch}/{EPOCHS} =====")
    steps_per_epoch = len(train_loader)
    for epoch in range(start_epoch, EPOCHS):
        train_sampler.set_epoch(epoch); model.train()
        sums, n_steps, t0 = defaultdict(float), 0, time.time()
        for i, (x, y, gen) in enumerate(tqdm(train_loader, desc=f"{loss_id} ep {epoch+1}/{EPOCHS}", leave=False)):
            x, y, gen = (t.to(DEVICE, non_blocking=True) for t in (x, y, gen))
            fake_codes = gen[gen != GEN_CODE["original"]]
            assert len(fake_codes) > 0 and torch.all(fake_codes == fake_codes[0])
            g = CATEGORIES[int(fake_codes[0])]
            lr = lr_at(epoch * steps_per_epoch + i, steps_per_epoch)
            parts, _ = train_step(state, opt, x, y, g, lr, loss_id)
            for k, v in parts.items(): sums[k] += v
            n_steps += 1
        val_table = metrics_table(frame_pred(val_df, predict_probs(state, ram_images("val", val_df), "val")))
        row = {"epoch": epoch + 1, "lr (end)": float(lr), "loss_id": loss_id}
        row.update({k: v / n_steps for k, v in sums.items()})
        row.update({f"m {SHORT[g]}": float(state.m[g]) for g in TRAIN_GENERATORS})
        if loss_id == "ARSD":
            row["m_real"] = float(state.m_real)
        row.update({"val video AUC": float(val_table["video AUC"].iloc[0]), "minutes": (time.time() - t0) / 60})
        history.append(row)
        save_atomic(state.save(), rd / f"epoch_{epoch+1:02d}.pt")
        save_atomic({"epoch": epoch + 1, "state": state.save(), "adam": opt.state_dict(), "history": history}, rd / "last.pt")
        print(" | ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in row.items()))
    if not (rd / "results.json").exists():
        state.proto = exact_prototypes(state)
        table = metrics_table(frame_pred(test_df, predict_probs(state, ram_images("test", test_df), "test")))
        display(table)
        res = {"architecture": "Arch4C", "notebook": "M16", "loss_id": loss_id,
               "deployed_trainable": P + 2,
               "test": table.to_dict(orient="records"), "history": history,
               "m": {SHORT[g]: float(state.m[g]) for g in TRAIN_GENERATORS}}
        if loss_id == "ARSD":
            res["m_real"] = float(state.m_real)
        for name, flag, index_path, root in [("dfd", RUN_DFD, DFD_INDEX, FACES_DFD), ("uadfv", RUN_UADFV, UADFV_INDEX, FACES_UADFV)]:
            if not flag: continue
            idx = pd.read_csv(index_path, dtype={"video_id": str})
            arr = load_paths([root / p for p in idx.path], name)
            res[f"{name}_video_auc"] = video_auc(idx, predict_probs(state, arr, name)); del arr
            print(name.upper(), res[f"{name}_video_auc"])
        (rd / "results.json").write_text(json.dumps(res, indent=1))
    del state, opt; torch.cuda.empty_cache()
    return rd

for lid in LOSS_IDS:
    train_one(lid)
print("M16 training finished for", LOSS_IDS)


===== M16 L0 dep=102,402 0/10 =====


epoch 1 | lr (end) 0.0002997 | loss_id L0 | loss -0.5597 | ce 0.3799 | margin 0.3022 | penalty 0.05357 | extra 0 | m DF 0.1315 | m F2F 0.1267 | m FS 0.143 | m NT 0.1104 | val video AUC 84.34 | minutes 7.969


epoch 2 | lr (end) 0.0002913 | loss_id L0 | loss -1.368 | ce 0.3024 | margin 0.003778 | penalty 0.05459 | extra 0 | m DF 0.1358 | m F2F 0.135 | m FS 0.1476 | m NT 0.1378 | val video AUC 85.73 | minutes 7.902


epoch 3 | lr (end) 0.0002661 | loss_id L0 | loss -1.453 | ce 0.2583 | margin 0.002736 | penalty 0.05552 | extra 0 | m DF 0.1384 | m F2F 0.142 | m FS 0.1499 | m NT 0.1436 | val video AUC 87.68 | minutes 7.933


epoch 4 | lr (end) 0.0002275 | loss_id L0 | loss -1.481 | ce 0.2315 | margin 0.003623 | penalty 0.05621 | extra 0 | m DF 0.1406 | m F2F 0.1484 | m FS 0.152 | m NT 0.1511 | val video AUC 93.01 | minutes 7.898


epoch 5 | lr (end) 0.0001802 | loss_id L0 | loss -1.498 | ce 0.2167 | margin 0.003424 | penalty 0.0568 | extra 0 | m DF 0.1443 | m F2F 0.1547 | m FS 0.1544 | m NT 0.1585 | val video AUC 91.47 | minutes 7.899


... (66 outputs left out)

dfd: 100%|██████████| 109454/109454 [01:20<00:00, 1366.06it/s]
                                                      

DFD 96.19775447972675


uadfv: 100%|██████████| 3136/3136 [00:01<00:00, 1641.34it/s]
                                                      

UADFV 99.62515618492294
M16 training finished for ['L0', 'PSC', 'CGSS', 'ARSD', 'BGTC']


## 6. Decision — which loss to KEEP


In [12]:
# 6.1 Decision — which loss to KEEP
def video_auc_from_records(records):
    t = pd.DataFrame(records).set_index("test set")
    row = {"seen": float(t.iloc[0]["video AUC"])}
    row.update({SHORT[g]: float(t.loc[g, "video AUC"]) for g in ALL_GENERATORS})
    return row

rows = [{"model": "A0 SB-LN+linear", "trainable": 36418, **A0, "UADFV": float("nan"), "keep_gate": False}]
for name, (folder, dfd, uadfv) in REFS.items():
    if not (folder / "results.json").exists():
        print("missing ref", name); continue
    res = json.loads((folder / "results.json").read_text())
    rows.append({"model": name, "trainable": res.get("deployed_trainable", P),
                 **video_auc_from_records(res["test"]),
                 "DFD": res.get("dfd_video_auc", dfd if dfd is not None else float("nan")),
                 "UADFV": res.get("uadfv_video_auc", uadfv if uadfv is not None else float("nan")),
                 "keep_gate": False})

sweep = []
for lid in LOSS_IDS:
    rd = run_dir(lid)
    if not (rd / "results.json").exists():
        print("missing", lid); continue
    res = json.loads((rd / "results.json").read_text())
    arch = {"model": f"Arch4C+{lid}", "trainable": res["deployed_trainable"],
            **video_auc_from_records(res["test"]),
            "DFD": res.get("dfd_video_auc", float("nan")),
            "UADFV": res.get("uadfv_video_auc", float("nan"))}
    gate = float(arch["NT"]) >= A0["NT"] and float(arch["DFD"]) >= 95.5
    arch["keep_gate"] = gate
    rows.append(arch)
    sweep.append(arch)

main_table = pd.DataFrame(rows).set_index("model")
display(main_table.round(2))
main_table.to_csv(M16_DIR / "main_table.csv")

# Rank passers: prefer higher NT, then higher DFD, then higher FSh
passers = [r for r in sweep if r["keep_gate"]]
if not passers:
    winner, reason = None, "no loss variant passed NT≥81.21 and DFD≥95.5"
else:
    passers = sorted(passers, key=lambda r: (r["NT"], r["DFD"], r["FSh"]), reverse=True)
    winner = passers[0]["model"]
    reason = f"best among gate-passers by (NT, DFD, FSh): {winner}"

print(reason)
if winner:
    print("DECISION: KEEP", winner)
else:
    print("DECISION: KEEP none — stick with M15c L0 only if it already passed, else re-tune kappa")

(M16_DIR / "decision.json").write_text(json.dumps({
    "notebook": "M16",
    "loss_ids": LOSS_IDS,
    "winner": winner,
    "reason": reason,
    "gate": {"NT_min": A0["NT"], "DFD_min": 95.5},
    "sweep": [{k: (bool(v) if k == "keep_gate" else v) for k, v in r.items()} for r in sweep],
}, indent=2))


,trainable,seen,DF,F2F,FS,NT,FSh,DFD,UADFV,keep_gate
model,,,,,,,,,,
A0 SB-LN+linear,36418,94.54,99.49,97.86,99.60,81.21,92.25,95.15,NaN,False
full LayerNorms (M4 λ=∞),102400,96.41,99.72,98.66,99.34,87.90,90.53,96.05,99.38,False
GBLA λ=0.01 (M4) / Arch1,102400,95.55,99.43,98.55,99.17,85.06,92.91,96.71,99.63,False
Arch4C L0 (M15c),102402,96.16,99.64,98.60,99.58,86.80,92.81,96.16,99.58,False
Arch4C+L0,102402,96.28,99.64,98.73,99.56,87.18,92.71,96.27,99.54,True
Arch4C+PSC,102402,96.58,99.71,98.63,99.58,88.39,92.46,96.52,98.92,True
Arch4C+CGSS,102402,95.80,99.70,98.56,99.62,85.32,94.37,96.29,99.71,True
Arch4C+ARSD,102402,96.04,99.59,98.45,99.48,86.65,93.16,96.33,99.58,True
Arch4C+BGTC,102402,96.62,99.71,98.54,99.47,88.74,92.53,96.20,99.63,True


best among gate-passers by (NT, DFD, FSh): Arch4C+BGTC
DECISION: KEEP Arch4C+BGTC


1655